# Notebook 8: Do matchers reward AI-polished resumes? (RQ5)

**Before running:** Runtime → Change runtime type → **T4 GPU**. Close any other GPU session first (Runtime → Manage sessions → Terminate).

**The question.** Many job seekers now run their resume through ChatGPT or a similar tool before applying. If a matcher scores the *polished* version higher than the original, with the *same facts*, then it rewards writing style, not fit. That would be unfair to people who don't use these tools.

**What this notebook does**
1. Picks **200 resumes** at random (fixed seed).
2. An LLM rewrites each one to sound more polished, **keeping every fact**. It never sees any job, so this is general polishing, not tailoring to a job. The rewriter is **Phi-4-mini-instruct** (Microsoft), a *different* model family from our judge (Qwen), so the judge can't simply prefer its own writing style.
3. Checks the rewrites automatically: new numbers, words copied from the job, length. You spot-check 20 by hand later (a sheet is saved).
4. Scores original vs polished with every method: TF-IDF, E5, BGE, MPNet (all pairs of these resumes), and the Qwen3-4B judge (one fit and one no-fit pair per resume).
5. Measures the score change, how many places a resume moves up in each job's ranking, and how often the judge's label changes.

**Resumable:** rewrites and judge scores are saved to Google Drive as they go. If Colab disconnects, just **Run all** again and it continues where it stopped.

**Run time:** about 1–1.5 hours (rewriting about 40–60 min, judge about 20 min, the rest about 10 min).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U transformers accelerate sentence-transformers openpyxl

import os, re, gc, time
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then run again'
DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
CACHE = f'{RESULTS}/llm_cache'
os.makedirs(CACHE, exist_ok=True)
SEED = 42
N_RESUMES = 200
MAX_WORDS = 900   # the same resume cut as the LLM judge in Notebook 4

all_df = pd.concat([pd.read_parquet(f'{DATA}/{s}.parquet').assign(split=s) for s in ('train', 'val', 'test')],
                   ignore_index=True)
g = all_df.groupby('job_id').agg(candidates=('resume_id', 'size'),
                                 relevant=('relevance', lambda r: int((r > 0).sum())),
                                 levels=('relevance', 'nunique'))
usable = g[(g.candidates >= 5) & (g.relevant >= 1) & (g.levels >= 2)].index
evald = all_df[all_df.job_id.isin(usable)].reset_index(drop=True)
evald['pair_key'] = evald.job_id + '|' + evald.resume_id

def cut(text, n_words):
    w = text.split()
    return text if len(w) <= n_words else ' '.join(w[:n_words])

rng = np.random.default_rng(SEED)
sample_ids = rng.choice(np.sort(evald.resume_id.unique()), N_RESUMES, replace=False)
res_text = evald.drop_duplicates('resume_id').set_index('resume_id').resume_text
orig = pd.DataFrame({'resume_id': sample_ids, 'original': [cut(res_text[r], MAX_WORDS) for r in sample_ids]})
print(f'{len(usable)} usable jobs, {len(evald)} pairs, {evald.resume_id.nunique()} resumes')
print(f'{len(orig)} resumes sampled; they appear in {evald.resume_id.isin(sample_ids).sum()} pairs')
print('original length (words):', orig.original.str.split().str.len().describe().round(0).to_dict())

## 1. Rewrite the resumes

The prompt asks for better wording, structure and action verbs, and forbids adding or removing any fact. Greedy decoding (no randomness), so a re-run gives the same text. A rewrite counts as **usable** if its length is between 0.6× and 1.8× the original and it wasn't cut off.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

REWRITER = 'microsoft/Phi-4-mini-instruct'
FALLBACK = 'Qwen/Qwen3-4B-Instruct-2507'   # only if Phi fails to load; then the judge shares its family (say so in the paper)
REWRITE_PATH = f'{RESULTS}/08_rewrites.parquet'
MAX_NEW = 1600

PROMPT = '''Rewrite the resume below so that it reads as polished and professional, the way a careful career coach or an AI writing assistant would improve it.

Rules:
- Keep every fact exactly: the same jobs, employers, job titles, dates, degrees, skills, tools and numbers.
- Do NOT add any skill, tool, certification, achievement, number or responsibility that is not in the original.
- Do NOT remove information.
- You may improve wording, grammar, structure, headings and bullet points, and use stronger action verbs.
- Output only the rewritten resume, with no comments before or after it.

Resume:
"""
{resume}
"""'''

def load_gen(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map='cuda')
    model.eval()
    return tok, model

def chat(tok, text):
    msgs = [{'role': 'user', 'content': PROMPT.format(resume=text)}]
    kwargs = {'tokenize': False, 'add_generation_prompt': True}
    try:
        return tok.apply_chat_template(msgs, enable_thinking=False, **kwargs)
    except TypeError:
        return tok.apply_chat_template(msgs, **kwargs)

def tidy(out):
    out = out.strip()
    out = re.sub(r"^(here is|here's|below is)[^\n]*\n", '', out, flags=re.I).strip()
    return out.strip('"').strip()

@torch.no_grad()
def rewrite_all(batch_size=6):
    done = pd.read_parquet(REWRITE_PATH) if os.path.exists(REWRITE_PATH) else pd.DataFrame()
    todo = orig[~orig.resume_id.isin(set(done.resume_id))] if len(done) else orig
    print(f'{len(done)} rewritten already, {len(todo)} to go')
    if len(todo) == 0:
        return done
    try:
        model_id = REWRITER; tok, model = load_gen(model_id)
    except Exception as e:
        print('Phi-4-mini failed to load, using the fallback. Error:', e)
        model_id = FALLBACK; tok, model = load_gen(model_id)
    # longest first, so a batch holds similar lengths and padding wastes less time
    todo = todo.assign(n=todo.original.str.len()).sort_values('n', ascending=False)
    for start in range(0, len(todo), batch_size):
        batch = todo.iloc[start:start + batch_size]
        enc = tok([chat(tok, t) for t in batch.original], return_tensors='pt', padding=True).to('cuda')
        t0 = time.perf_counter()
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tok.pad_token_id)
        sec = (time.perf_counter() - t0) / len(batch)
        new_tok = out[:, enc.input_ids.shape[1]:]
        texts = tok.batch_decode(new_tok, skip_special_tokens=True)
        n_gen = (new_tok != tok.pad_token_id).sum(dim=1).tolist()
        rows = [{'resume_id': rid, 'polished': tidy(t), 'rewriter': model_id, 'sec_per_resume': sec,
                 'hit_limit': n >= MAX_NEW}
                for rid, t, n in zip(batch.resume_id, texts, n_gen)]
        done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
        done.to_parquet(REWRITE_PATH, index=False)
        print(f'  {len(done)}/{len(orig)} rewritten ({sec:.0f} s per resume in this batch)')
    del model; gc.collect(); torch.cuda.empty_cache()
    return done

rw = rewrite_all().merge(orig, on='resume_id')
rw['len_ratio'] = rw.polished.str.split().str.len() / rw.original.str.split().str.len()
rw['usable'] = rw.len_ratio.between(0.6, 1.8) & ~rw.hit_limit & (rw.polished.str.len() > 0)
print('rewriter:', rw.rewriter.unique())
print(f"usable rewrites: {rw.usable.sum()} of {len(rw)}")
print('length ratio (polished / original):', rw.len_ratio.describe().round(2).to_dict())

## 2. Did the rewrite keep the facts?

Two automatic checks, plus a sheet for your hand check.
- **New numbers:** any number in the polished text that is not in the original (a new year, "5 years", "30%"). A rewrite with a new number may have invented a fact.
- **Job words added:** for each job–resume pair, words from that job's description that were *not* in the original resume but appear in the polished one. The rewriter never saw the job, so these are general words ("management", "communication"), but they are exactly what keyword matchers count.

The **strict** subset (usable and no new numbers) is used for a sensitivity check in section 5.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def words(t):
    return {w for w in re.findall(r'[a-z][a-z0-9+#]*', t.lower()) if len(w) >= 3 and w not in ENGLISH_STOP_WORDS}
def numbers(t):
    return set(re.findall(r'\d+(?:\.\d+)?', t.replace(',', '')))

rw['new_numbers'] = [sorted(numbers(p) - numbers(o)) for o, p in zip(rw.original, rw.polished)]
rw['n_new_numbers'] = rw.new_numbers.str.len()
rw['strict'] = rw.usable & (rw.n_new_numbers == 0)
print(f"rewrites with a new number: {(rw.usable & (rw.n_new_numbers > 0)).sum()} of {rw.usable.sum()} usable")
print(f"strict subset: {rw.strict.sum()} resumes")

# Hand-check sheet: 20 usable rewrites, side by side
spot = rw[rw.usable].sample(min(20, rw.usable.sum()), random_state=SEED)
sheet = pd.DataFrame({'resume_id': spot.resume_id, 'original': spot.original, 'polished': spot.polished,
                      'new numbers (auto)': spot.new_numbers.map(', '.join),
                      'facts kept? (yes / no)': '', 'what was added or lost': ''})
sheet_path = f'{RESULTS}/08_spotcheck.xlsx'
sheet.to_excel(sheet_path, index=False)
from openpyxl import load_workbook
from openpyxl.styles import Alignment
wb = load_workbook(sheet_path); ws = wb.active
for col, width in zip('ABCDEF', (14, 70, 70, 16, 14, 30)):
    ws.column_dimensions[col].width = width
for row in ws.iter_rows(min_row=2):
    for c in row:
        c.alignment = Alignment(wrap_text=True, vertical='top')
ws.freeze_panes = 'B2'
wb.save(sheet_path)
print('hand-check sheet saved:', sheet_path)

ex = rw[rw.usable].iloc[0]
print('\n===== EXAMPLE: ORIGINAL (first 120 words) =====\n', ' '.join(ex.original.split()[:120]))
print('\n===== EXAMPLE: POLISHED (first 120 words) =====\n', ' '.join(ex.polished.split()[:120]))

## 3. Score original vs polished: TF-IDF and embeddings

All candidates of every job keep their **original** resumes; only the sampled resume is swapped for its polished version. So we can see both the score change and **how many places it moves up** in that job's ranking.

The embedding models read at most 512 tokens (about 380 words), the same as in Notebook 3b, so a rewrite that moves strong content to the top can matter.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

usable_ids = set(rw.resume_id[rw.usable])
pol_text = rw.set_index('resume_id').polished
evald['resume_cut'] = evald.resume_text.map(lambda t: cut(t, MAX_WORDS))
target = evald[evald.resume_id.isin(usable_ids)].copy()
target['polished'] = target.resume_id.map(pol_text)
jw = {jid: words(t) for jid, t in evald.drop_duplicates('job_id')[['job_id', 'job_text']].itertuples(index=False)}
ow = {r: words(o) for r, o in zip(rw.resume_id, rw.original)}
pw = {r: words(p) for r, p in zip(rw.resume_id, rw.polished)}
target['job_words_added'] = [len((jw[j] & pw[r]) - ow[r]) for j, r in zip(target.job_id, target.resume_id)]
target['job_words_lost'] = [len((jw[j] & ow[r]) - pw[r]) for j, r in zip(target.job_id, target.resume_id)]
print(f'{len(target)} pairs to score ({target.resume_id.nunique()} resumes, {target.job_id.nunique()} jobs)')
print('job words added per pair:', target.job_words_added.describe().round(1).to_dict())

# TF-IDF fitted on the original corpus only (as in Notebook 3b); rows are L2-normalised, so a dot product is the cosine
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=100_000)
tfidf.fit(list(evald.job_text.unique()) + list(evald.resume_cut.unique()))
def tfidf_cos(jobs, resumes):
    return np.asarray(tfidf.transform(jobs).multiply(tfidf.transform(resumes)).sum(axis=1)).ravel()
evald['tfidf'] = tfidf_cos(evald.job_text, evald.resume_cut)
target['tfidf_pol'] = tfidf_cos(target.job_text, target.polished)

MODELS = [('e5', 'intfloat/e5-base-v2', 'query: ', 'passage: '),
          ('bge', 'BAAI/bge-base-en-v1.5', '', ''),
          ('mpnet', 'sentence-transformers/all-mpnet-base-v2', '', '')]
for col, hf_id, qp, dp in MODELS:
    m = SentenceTransformer(hf_id, device='cuda'); m.max_seq_length = 512
    enc = lambda texts, p: m.encode([p + t for t in texts], batch_size=32, normalize_embeddings=True, show_progress_bar=False)
    jobs_u = evald.drop_duplicates('job_id'); res_u = evald.drop_duplicates('resume_id')
    je = dict(zip(jobs_u.job_id, enc(jobs_u.job_text.tolist(), qp)))
    ro = dict(zip(res_u.resume_id, enc(res_u.resume_cut.tolist(), dp)))
    rp = dict(zip(rw.resume_id[rw.usable], enc(rw.polished[rw.usable].tolist(), dp)))
    evald[col] = [float(je[j] @ ro[r]) for j, r in zip(evald.job_id, evald.resume_id)]
    target[col + '_pol'] = [float(je[j] @ rp[r]) for j, r in zip(target.job_id, target.resume_id)]
    del m; gc.collect(); torch.cuda.empty_cache()
    print(col, 'done')

CONTENT = ['tfidf', 'e5', 'bge', 'mpnet']
for col in CONTENT:
    target[col + '_orig'] = target.pair_key.map(evald.set_index('pair_key')[col])

# Rank of the sampled resume within its job (1 = top), before and after, other candidates unchanged
for col in CONTENT:
    by_job = evald.groupby('job_id')[col].apply(np.array).to_dict()
    def rank_of(jid, s):
        return 1 + int((by_job[jid] > s).sum())
    target[col + '_rank_orig'] = [rank_of(j, s) for j, s in zip(target.job_id, target[col + '_orig'])]
    # the job's list still holds the original score of this resume; remove it once before ranking the polished one
    target[col + '_rank_pol'] = [rank_of(j, p) - int(o > p) for j, o, p in zip(target.job_id, target[col + '_orig'], target[col + '_pol'])]
    target[col + '_places_up'] = target[col + '_rank_orig'] - target[col + '_rank_pol']
target['candidates'] = target.job_id.map(evald.groupby('job_id').size())
print('scored')

## 4. Score original vs polished: the Qwen3-4B judge

The same prompt and scoring as Notebook 4 (letter probabilities, score = 2·P(Good) + P(Potential)). To keep the run short, one **fit** pair and one **no-fit** pair per resume (fixed seed), each scored twice: original and polished.

In [ ]:
SYSTEM = ('You are an experienced technical recruiter. You judge how well a candidate fits one '
          'specific job, based only on the job description and the resume.')
TEMPLATE = '''Job description:
"""
{job}
"""

Candidate resume:
"""
{resume}
"""

How well does this candidate fit THIS job?
A) Good Fit - meets the main requirements (role, key skills, experience level)
B) Potential Fit - related background, but missing some key requirements
C) No Fit - different field or clearly unqualified

Answer with a single letter: A, B or C.'''

def cut_dots(text, n_words):
    w = text.split()
    return text if len(w) <= n_words else ' '.join(w[:n_words]) + ' ...'

def build_prompt(tok, job, resume):
    msgs = [{'role': 'system', 'content': SYSTEM},
            {'role': 'user', 'content': TEMPLATE.format(job=cut_dots(job, 450), resume=cut_dots(resume, 900))}]
    kwargs = {'tokenize': False, 'add_generation_prompt': True}
    try:
        return tok.apply_chat_template(msgs, enable_thinking=False, **kwargs)
    except TypeError:
        return tok.apply_chat_template(msgs, **kwargs)

def load_judge(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map='cuda')
    model.eval()
    letter_ids = [tok.encode(x, add_special_tokens=False)[0] for x in ('A', 'B', 'C')]
    assert len(set(letter_ids)) == 3
    return tok, model, letter_ids

@torch.no_grad()
def score_pairs(model_id, tag, pairs, batch_size=4, save_every=100):
    path = f'{CACHE}/{tag}.parquet'
    empty = pd.DataFrame({'pair_key': pd.Series(dtype=str), **{c: pd.Series(dtype=float) for c in ('pA', 'pB', 'pC', 'ms')}})
    done = pd.read_parquet(path) if os.path.exists(path) else empty
    todo = pairs[~pairs.pair_key.isin(set(done.pair_key))]
    print(f'{tag}: {len(done)} cached, {len(todo)} to score')
    if len(todo) == 0:
        return done
    tok, model, letter_ids = load_judge(model_id)
    new_rows = []
    for start in range(0, len(todo), batch_size):
        batch = todo.iloc[start:start + batch_size]
        prompts = [build_prompt(tok, j, r) for j, r in zip(batch.job_text, batch.resume_text)]
        enc = tok(prompts, return_tensors='pt', padding=True).to('cuda')
        torch.cuda.synchronize(); t0 = time.perf_counter()
        logits = model(**enc).logits[:, -1, :].float()
        torch.cuda.synchronize(); ms = (time.perf_counter() - t0) * 1000 / len(batch)
        probs = torch.softmax(logits[:, letter_ids], dim=-1).cpu().numpy()
        if np.isnan(probs).any():
            raise RuntimeError('NaN in model output (fp16 overflow).')
        for key, p in zip(batch.pair_key, probs):
            new_rows.append({'pair_key': key, 'pA': p[0], 'pB': p[1], 'pC': p[2], 'ms': ms})
        if len(new_rows) >= save_every or start + batch_size >= len(todo):
            done = pd.concat([done, pd.DataFrame(new_rows)], ignore_index=True)
            done.to_parquet(path, index=False)
            new_rows = []
            print(f'  {len(done)}/{len(pairs)} pairs scored')
    del model; gc.collect(); torch.cuda.empty_cache()
    return done

pick_rng = np.random.default_rng(SEED)
picked = []
for rid, grp in target.groupby('resume_id', sort=True):
    for want_fit in (True, False):
        c = grp[(grp.relevance > 0) == want_fit]
        if len(c):
            picked.append(c.iloc[int(pick_rng.integers(len(c)))])
J = pd.DataFrame(picked).reset_index(drop=True)
long = pd.concat([J.assign(pair_key=J.pair_key + '|orig', resume_text=J.resume_cut),
                  J.assign(pair_key=J.pair_key + '|pol', resume_text=J.polished)], ignore_index=True)
print(f'judge pairs: {len(J)} ({(J.relevance > 0).sum()} fit, {(J.relevance == 0).sum()} no-fit), {len(long)} calls')
jscores = score_pairs('Qwen/Qwen3-4B-Instruct-2507', 'nb8-qwen3-4b-polish', long).set_index('pair_key')

LABELS = np.array(['Good Fit', 'Potential Fit', 'No Fit'])
for v in ('orig', 'pol'):
    s = jscores.loc[J.pair_key + '|' + v]
    J['qwen4b_' + v] = (2 * s.pA + s.pB).to_numpy()
    J['qwen4b_label_' + v] = LABELS[s[['pA', 'pB', 'pC']].to_numpy(dtype=float).argmax(axis=1)]
print('judged')

## 5. Results

For each method: the mean change in score (polished − original) with a 95% bootstrap CI over **resumes** (pairs of the same resume move together), the change in **standard deviations** of that method's original scores (so methods can be compared), the share of pairs that went up, and a Wilcoxon signed-rank p-value with Holm correction across the five methods.

**How to read it:** a positive, significant change means the method gives a higher score to the same facts written more smoothly. The most worrying case is a rise for **no-fit** pairs, because then polishing pushes unsuitable candidates up the list.

In [ ]:
def holm(p):
    p = np.asarray(p, dtype=float); order = np.argsort(p); k = len(p); adj = np.empty(k); running = 0
    for rank, i in enumerate(order):
        running = max(running, (k - rank) * p[i]); adj[i] = min(1, running)
    return adj

def cluster_ci(d, groups, n=2000):
    df = pd.DataFrame({'d': d, 'g': groups}); sums = df.groupby('g').d.agg(['sum', 'count'])
    rng = np.random.default_rng(SEED); ids = np.arange(len(sums)); out = []
    for _ in range(n):
        b = sums.iloc[rng.choice(ids, len(ids))]
        out.append(b['sum'].sum() / b['count'].sum())
    return np.percentile(out, [2.5, 97.5])

def effect_table(subset_name, keep_ids):
    rows = []
    sources = [(c, target, c + '_orig', c + '_pol', evald[c].std()) for c in CONTENT]
    sources.append(('qwen4b judge', J, 'qwen4b_orig', 'qwen4b_pol', J.qwen4b_orig.std()))
    for name, df, o, p, sd in sources:
        df = df[df.resume_id.isin(keep_ids)]
        for part, mask in [('all pairs', np.ones(len(df), bool)), ('fit pairs', df.relevance.to_numpy() > 0),
                           ('no-fit pairs', df.relevance.to_numpy() == 0)]:
            x = df[mask]; d = (x[p] - x[o]).to_numpy()
            if len(d) < 10:
                continue
            lo, hi = cluster_ci(d, x.resume_id.to_numpy())
            row = {'subset': subset_name, 'method': name, 'pairs': part, 'n': len(d), 'resumes': x.resume_id.nunique(),
                   'mean change': round(d.mean(), 4), 'CI': f'[{lo:.4f}, {hi:.4f}]', 'change in SDs': round(d.mean() / sd, 3),
                   'share up': round((d > 0).mean(), 3), 'p': wilcoxon(d).pvalue if np.any(d != 0) else 1.0}
            if name in CONTENT:
                row['mean places up'] = round(x[name + '_places_up'].mean(), 2)
                row['share reaching top 5'] = round(((x[name + '_rank_orig'] > 5) & (x[name + '_rank_pol'] <= 5)).mean(), 3)
            rows.append(row)
    t = pd.DataFrame(rows)
    t['p (Holm)'] = t.groupby('pairs').p.transform(lambda s: holm(s.to_numpy()))   # across the five methods, per pair type
    return t

main = effect_table('usable', set(rw.resume_id[rw.usable]))
strict = effect_table('strict (no new numbers)', set(rw.resume_id[rw.strict]))
pd.set_option('display.width', 250)
main

In [ ]:
# Sensitivity: the same table on rewrites with no new numbers. The conclusion should not change.
strict

In [ ]:
# Judge labels before (rows) and after (columns) polishing, by the dataset label
for part, mask in [('dataset fit pairs', J.relevance > 0), ('dataset no-fit pairs', J.relevance == 0)]:
    x = J[mask & J.resume_id.isin(rw.resume_id[rw.usable])]
    ct = pd.crosstab(x.qwen4b_label_orig, x.qwen4b_label_pol).reindex(index=LABELS, columns=LABELS, fill_value=0)
    up = (x.qwen4b_label_orig.map({'No Fit': 0, 'Potential Fit': 1, 'Good Fit': 2}) <
          x.qwen4b_label_pol.map({'No Fit': 0, 'Potential Fit': 1, 'Good Fit': 2})).mean()
    down = (x.qwen4b_label_orig.map({'No Fit': 0, 'Potential Fit': 1, 'Good Fit': 2}) >
            x.qwen4b_label_pol.map({'No Fit': 0, 'Potential Fit': 1, 'Good Fit': 2})).mean()
    print(f'\n{part} ({len(x)}): label moved UP for {up:.1%}, DOWN for {down:.1%}')
    print(ct)

# Is the change explained by job words the rewrite added? (Spearman, all pairs)
from scipy.stats import spearmanr
for c in CONTENT:
    rho, p = spearmanr(target.job_words_added, target[c + '_pol'] - target[c + '_orig'])
    print(f'{c:6} change vs job words added: rho = {rho:+.3f} (p = {p:.3g})')

## 6. Save

In [ ]:
main.to_csv(f'{RESULTS}/08_polish_effects.csv', index=False)
strict.to_csv(f'{RESULTS}/08_polish_effects_strict.csv', index=False)
keep = ['job_id', 'resume_id', 'pair_key', 'relevance', 'split', 'candidates', 'job_words_added', 'job_words_lost'] + \
       [c + s for c in CONTENT for s in ('_orig', '_pol', '_rank_orig', '_rank_pol', '_places_up')]
target[keep].to_parquet(f'{RESULTS}/08_pair_scores.parquet', index=False)
J.drop(columns=['job_text', 'resume_text', 'resume_cut', 'polished'], errors='ignore').to_parquet(f'{RESULTS}/08_judge_scores.parquet', index=False)
rw[['resume_id', 'rewriter', 'sec_per_resume', 'hit_limit', 'len_ratio', 'usable', 'n_new_numbers', 'strict']].to_csv(
    f'{RESULTS}/08_rewrite_checks.csv', index=False)
print('saved. Rewriting took about', round(rw.sec_per_resume.mean(), 1), 's per resume.')
print('Next: download this notebook (File > Download > .ipynb). '
      'Later, download results/08_spotcheck.xlsx from Drive and fill in the two empty columns.')